In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import tiktoken
import urllib.request
import os

# 1. Load Data
if not os.path.exists('kobzar.txt'):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/jimchen2/kobzar-gpt/refs/heads/main/kobzar.txt", "kobzar.txt")

with open('kobzar.txt', 'r', encoding='utf-8') as f:
    full_text = f.read()

def get_batch(split):
    # Generates a small batch of data of inputs x and targets y
    d = train_data if split == 'train' else val_data
    # Grab random starting indexes
    ix = torch.randint(len(d) - T, (B,))

    # x is the input [B, T], y is the target shifted by 1 [B, T]
    x = torch.stack([d[i:i+T] for i in ix])
    y = torch.stack([d[i+1:i+T+1] for i in ix])
    return x.to(DEVICE), y.to(DEVICE)

In [ ]:
class Head(nn.Module):
    """ One head of self-attention """
    def __init__(self, head_size):
        super().__init__()
        # W_Q, W_K, W_V matrices (Linear layers without bias usually)
        self.key = nn.Linear(D, head_size, bias=False)
        self.query = nn.Linear(D, head_size, bias=False)
        self.value = nn.Linear(D, head_size, bias=False)

        # Cause-al Masking: Lower triangular matrix (tril)
        # We register it as a buffer so PyTorch doesn't treat it as a trained weight
        self.register_buffer('tril', torch.tril(torch.ones(T, T)))

    def forward(self, x):
        B, T_seq, C = x.shape
        k = self.key(x)   # [B, T, D/H]
        q = self.query(x) # [B, T, D/H]

        # Attention scores: (Q * K^T) / sqrt(D/H)
        # transpose(-2, -1) flips the last two dimensions to match [B, D/H, T]
        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)

        # Apply the Cause-al Mask (values above diagonal become -inf)
        wei = wei.masked_fill(self.tril[:T_seq, :T_seq] == 0, float('-inf'))

        # Softmax applies across rows
        wei = F.softmax(wei, dim=-1)

        v = self.value(x) # [B, T, D/H]
        out = wei @ v     # [B, T, D/H]
        return out

class MultiHeadAttention(nn.Module):
    """ Multiple heads of self-attention running in parallel """
    def __init__(self, num_heads, head_size):
        super().__init__()
        # Concat(head_1, ..., head_H)
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        # Weight Matrix W_O
        self.proj = nn.Linear(num_heads * head_size, D)

    def forward(self, x):
        # Run all heads in parallel and concatenate along the last dimension
        out = torch.cat([h(x) for h in self.heads], dim=-1) # [B, T, D]
        out = self.proj(out)                                # [B, T, D]
        return out

class FeedForward(nn.Module):
    """ Part B: Up Projection, Activation, Down Projection """
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd)
        )

    def forward(self, x):
        return self.net(x) # [B, T, D]

class Block(nn.Module):
    """ Transformer block: Attention + Feed Forward """
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        # LayerNorm applied across rows
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        # x = x + Attention(LayerNorm(x))
        x = x + self.sa(self.ln1(x))
        # x = x + FeedForward(LayerNorm(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [ ]:
class ClassicGPT(nn.Module):
    def __init__(self):
        super().__init__()
        # Input: TokenEmbeddings + PositionalEmbeddings
        self.token_embedding_table = nn.Embedding(V, D)
        self.position_embedding_table = nn.Embedding(T, D)

        # N Blocks of Transformer
        self.blocks = nn.Sequential(*[Block(D, n_head=H) for _ in range(N)])

        # Output LayerNorm
        self.ln_f = nn.LayerNorm(D)

        # Un-embedding Matrix (Logits)
        self.lm_head = nn.Linear(D, V)

    def forward(self, idx, targets=None):
        idx = idx[:, -T:]
        if targets is not None:
            targets = targets[:, -T:] # Must crop targets to match!

        B, T_seq = idx.shape

        # Get embeddings
        tok_emb = self.token_embedding_table(idx) # [B, T_seq, D]
        pos_emb = self.position_embedding_table(torch.arange(T_seq, device=DEVICE)) # [T_seq, D]

        # X = TokenEmbeddings + PositionalEmbeddings
        x = tok_emb + pos_emb # [B, T_seq, D]

        # Pass through Transformer blocks
        x = self.blocks(x)    # [B, T_seq, D]
        x = self.ln_f(x)      # [B, T_seq, D]

        # Logits [B, T_seq, V]
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            # Reshape to calculate Loss
            B, T_seq, V_size = logits.shape
            logits_reshaped = logits.view(B * T_seq, V_size)
            targets_reshaped = targets.view(B * T_seq)

            loss = F.cross_entropy(logits_reshaped, targets_reshaped)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # Inference: Predict next token based on sampling
        for _ in range(max_new_tokens):

            logits, _ = self(idx)

            # Focus only on the last time step (Logits_{T-1})
            logits = logits[:, -1, :] # [B, V]

            # Apply softmax
            probs = F.softmax(logits, dim=-1) # [B, V]

            # Sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # [B, 1]

            # Append to the sequence
            idx = torch.cat((idx, idx_next), dim=1) # [B, T+1]

        return idx

In [ ]:

B = 8          # Batch Size
T = 32         # Context length (Input Tokens)
D = 256        # Embedding Dimension
H = 8          # Number of Attention Heads
N = 12          # Number of Transformer Blocks
LR = 1e-3
MAX_ITERS = 3000
TOKENIZER = "o200k_base"
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Tokenize
enc = tiktoken.get_encoding(TOKENIZER)
tokens = enc.encode(full_text)
data = torch.tensor(tokens, dtype=torch.long)
V = enc.n_vocab # Tokenizer Size (~200,000)

print(f"Total tokens in Kobzar: {len(data)}")
print(f"Vocabulary Size (V): {V}")

# Split into train/val
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]


Total tokens in Kobzar: 209951
Vocabulary Size (V): 200019


In [ ]:
model = ClassicGPT().to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

print("Starting training...")
for iter in range(MAX_ITERS):
    # Get a batch
    xb, yb = get_batch('train')

    # Forward pass
    logits, loss = model(xb, yb)

    # Backward pass
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if iter % 100 == 0:
        print(f"Step {iter}: Loss = {loss.item():.4f}")

Starting training...
Step 0: Loss = 12.4323
Step 100: Loss = 7.5231
Step 200: Loss = 6.7541
Step 300: Loss = 6.3842
Step 400: Loss = 5.9448
Step 500: Loss = 5.3305
Step 600: Loss = 5.5256
Step 700: Loss = 5.4239
Step 800: Loss = 4.8414
Step 900: Loss = 4.7682
Step 1000: Loss = 5.2111
Step 1100: Loss = 4.6835
Step 1200: Loss = 4.7211
Step 1300: Loss = 4.7238
Step 1400: Loss = 4.7490
Step 1500: Loss = 4.9537
Step 1600: Loss = 4.8958
Step 1700: Loss = 4.5136
Step 1800: Loss = 4.4383
Step 1900: Loss = 4.4944
Step 2000: Loss = 4.5327
Step 2100: Loss = 4.0476
Step 2200: Loss = 4.3393
Step 2300: Loss = 4.2061
Step 2400: Loss = 4.1438
Step 2500: Loss = 3.7742
Step 2600: Loss = 4.6070
Step 2700: Loss = 4.1524
Step 2800: Loss = 3.8603
Step 2900: Loss = 3.9520


In [ ]:
print("\n--- Generating Text ---")
# Start with a simple blank context (token 0 or simply a newline)
context = torch.zeros((1, 1), dtype=torch.long, device=DEVICE)

# Generate 50 tokens
generated_tokens = model.generate(context, max_new_tokens=50)[0].tolist()

# Decode back to text
generated_text = enc.decode(generated_tokens)
print(generated_text)


--- Generating Text ---
! Не знову розліпала… за не тілько колись І згадаю сіли що. Як лист, що ледащо царі очі свої діти, І тліс частом Лебирали.


In [ ]:
prompts = [
    "іде",
    "Україна",
    "Як умру",
]

print("\n--- Generating Text ---")
model.eval() # Set model to inference mode

# Disable gradients to save memory and speed up generation
with torch.no_grad():
    for text in prompts:
        # 1. Encode using your tiktoken tokenizer
        input_ids = enc.encode(text)

        # 2. Convert to tensor with batch dimension [1, Seq_Len]
        context = torch.tensor([input_ids], dtype=torch.long, device=DEVICE)

        # 3. Generate 50 new tokens
        generated_idx = model.generate(context, max_new_tokens=50)

        # 4. Decode the output back to text
        generated_text = enc.decode(generated_idx[0].tolist())

        print(f"Prompt: '{text}'")
        print(f"Result:\n{generated_text}")
        print("-" * 40)

In [ ]:
# --- Save the Model Weights ---
save_path = "kobzar_gpt_weights.pth"
torch.save(model.state_dict(), save_path)
print(f"Model weights saved to {save_path}")

Model weights saved to kobzar_gpt_weights.pth
